<a href="https://colab.research.google.com/github/MartVASS/MaskArchitectureAnomaly_CourseProject/blob/main/Step8_temp_cs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/FAIML_DANIELE_drive//MaskArchitectureAnomaly_CourseProject
!pip install -r eomt/requirements.txt

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject


In [ ]:


import os, glob

DATA_ROOT = "data/Validation_Dataset"
print(os.listdir(DATA_ROOT))

sample_images = glob.glob(f"{DATA_ROOT}/RoadAnomaly21/images/*")
print(len(sample_images), sample_images[:3])

['.DS_Store', 'FS_LostFound_full', 'fs_static', 'RoadAnomaly', 'RoadObsticle21', 'RoadAnomaly21']
10 ['data/Validation_Dataset/RoadAnomaly21/images/5.png', 'data/Validation_Dataset/RoadAnomaly21/images/1.png', 'data/Validation_Dataset/RoadAnomaly21/images/6.png']


In [ ]:
import sys, yaml
sys.path.append("eomt")

config_path = "eomt/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

config["trainer"]["logger"]["init_args"]["name"]

'cityscapes_semantic_eomt_base_640'

In [ ]:
from huggingface_hub import hf_hub_download

name = config["trainer"]["logger"]["init_args"]["name"]

ckpt_path = hf_hub_download(
    repo_id=f"S362484/{name}",
    filename="eomt_cityscapes.bin",
)

print(ckpt_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


/root/.cache/huggingface/hub/models--S362484--cityscapes_semantic_eomt_base_640/snapshots/0a0d8a920f846a2ba1347377f55a7fc2daa17a50/eomt_cityscapes.bin


In [ ]:
import os, sys, yaml, warnings, importlib
import torch
from torch.nn import functional as F
from lightning import seed_everything

seed_everything(0, verbose=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

PROJECT_ROOT = "/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject"
EOMT_ROOT = f"{PROJECT_ROOT}/eomt"

%cd {EOMT_ROOT}

if EOMT_ROOT not in sys.path:
    sys.path.append(EOMT_ROOT)

config_path = "configs/dinov2/cityscapes/semantic/eomt_base_640.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

img_size = (1024, 1024)
num_classes = 19

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)

encoder = encoder_cls(
    img_size=img_size,
    **encoder_cfg.get("init_args", {})
)

# Load network
network_cfg = config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)

network_kwargs = {
    k: v for k, v in network_cfg["init_args"].items()
    if k != "encoder"
}

network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module wrapper
lit_module_name, lit_class_name = config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)

model_kwargs = {
    k: v for k, v in config["model"]["init_args"].items()
    if k != "network"
}

model = lit_cls(
    img_size=img_size,
    num_classes=num_classes,
    network=network,
    **model_kwargs,
).eval().to(device)

# Load pretrained checkpoint
state_dict = torch.load(
    ckpt_path,
    map_location=device,
    weights_only=False,
)

missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("Model loaded")
print("missing keys:", len(missing))
print("unexpected keys:", len(unexpected))

device: cuda
/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject/eomt


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Model loaded
missing keys: 0
unexpected keys: 0


In [ ]:

import shutil

shutil.rmtree("saved_eomt_predictions_temp_v2", ignore_errors=True)
open("results_eomt_temperature_all.txt", "w").close()

In [ ]:
%cd /content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject

!python eval/eval_eomt_temp.py \
  --datasets RoadAnomaly RoadAnomaly21 fs_static LostFound RoadObsticle21 \
  --methods  temperature \
  --temperatures 0.5 0.75 1.1 \
  --results-file results_eomt_temperature_all.txt

/content/drive/MyDrive/FAIML_DANIELE_drive/MaskArchitectureAnomaly_CourseProject
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['network'])`.
Loaded EoMT checkpoint: /root/.cache/huggingface/hub/models--S362484--cityscapes_semantic_eomt_base_640/snapshots/0a0d8a920f846a2ba1347377f55a7fc2daa17a50/eomt_cityscapes.bin
Missing keys: 0 | Unexpected keys: 0

========== DATASET: RoadAnomaly ==========
data/Validation_Dataset/RoadAnomaly/images/0.jpg
data/Validation_Dataset/RoadAnomaly/images/1.jpg
data/Validation_Dataset/RoadAnomaly/images/10.jpg
data/Validation_Dataset/RoadAnomaly/images/11.jpg
data/Validation_Dataset/RoadAnomaly/images/12.jpg
data/Validation_Dataset/RoadAnomaly/images/13.jpg
data/Validation_Dataset/RoadAnomaly/images/14.jpg
data/Validation_Dataset/RoadAnomaly/images/15.jpg


In [ ]:
with open("results_eomt_temperature_all.txt", "r") as f:
    print(f.read())


EoMT anomaly evaluation | checkpoint=cityscapes_semantic_eomt_base_640
   cityscapes_semantic_eomt_base_640       RoadAnomaly  temperature  T=0.5      AUPRC: 74.7413  FPR@TPR95: 20.6404
   cityscapes_semantic_eomt_base_640       RoadAnomaly  temperature  T=0.75     AUPRC: 75.1618  FPR@TPR95: 20.0826
   cityscapes_semantic_eomt_base_640       RoadAnomaly  temperature  T=1.1      AUPRC: 75.1315  FPR@TPR95: 17.9856
   cityscapes_semantic_eomt_base_640       RoadAnomaly  temperature  BEST_T=0.75     AUPRC: 75.1618  FPR@TPR95: 20.0826
   cityscapes_semantic_eomt_base_640     RoadAnomaly21  temperature  T=0.5      AUPRC: 73.6372  FPR@TPR95: 53.6759
   cityscapes_semantic_eomt_base_640     RoadAnomaly21  temperature  T=0.75     AUPRC: 72.8986  FPR@TPR95: 51.2799
   cityscapes_semantic_eomt_base_640     RoadAnomaly21  temperature  T=1.1      AUPRC: 74.2890  FPR@TPR95: 29.9995
   cityscapes_semantic_eomt_base_640     RoadAnomaly21  temperature  BEST_T=1.1      AUPRC: 74.2890  FPR@TPR95: 29.999